# Flychess FlyNet: from-scratch training

This notebook is the reproducible hosted-runtime path for FlyNet. It creates a new legal-position dataset, generates a new sparse graph, initializes every trainable tensor from a seed, trains with Stockfish labels, validates the held-out split, and packages a model-only release. No existing neural checkpoint is loaded.

The phrase **100× better** is treated as a benchmark target. The release is not promoted on that basis until the fixed evaluation suite records the comparison metrics.

In [ ]:
from pathlib import Path
import subprocess

project_dir = Path('/content/flychess')
if (project_dir / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=project_dir, check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/EF-Code/flychess.git', str(project_dir)], check=True)
print('source checkout ready')

In [ ]:
%cd /content/flychess
!python -m pip install -q -e '.[flynet,dev]'
!sudo apt-get update -qq
!sudo apt-get install -y -qq stockfish
!python -m pytest -q

In [ ]:
from google.colab import userdata
assert userdata.get('HF_TOKEN'), 'HF_TOKEN is missing from Colab secrets'
assert userdata.get('GITHUB_ACCESS_TOKEN'), 'GITHUB_ACCESS_TOKEN is missing from Colab secrets'
print('required secrets are available; values are not displayed')

## 1. Generate labels

The sampler creates legal positions with a deterministic seed. Stockfish contributes only the target move and a bounded value label.

In [ ]:
!python scripts/generate_flynet_dataset.py \
  --engine stockfish \
  --depth 3 \
  --samples 20000 \
  --seed 20260917 \
  --min-plies 4 \
  --max-plies 60 \
  --output /content/flynet-dataset.npz

## 2. Train the independent graph model

The graph seed and weight seed are recorded separately. The release config must report `pretrained_weights_used: false`.

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('training device:', device)
!python scripts/train_flynet.py \
  --dataset /content/flynet-dataset.npz \
  --output-dir /content/flynet-release \
  --seed 20260917 \
  --graph-seed 20260918 \
  --nodes 4096 \
  --edges 200000 \
  --input-nodes 256 \
  --readout-nodes 320 \
  --steps 6 \
  --readout-width 256 \
  --epochs 20 \
  --batch-size 256 \
  --device {device}

In [ ]:
!python scripts/evaluate_flynet.py \
  --dataset /content/flynet-dataset.npz \
  --weights /content/flynet-release/flynet.safetensors \
  --graph /content/flynet-release/flynet-graph.npz \
  --graph-metadata /content/flynet-release/flynet-graph.json \
  --config /content/flynet-release/flynet-config.json

In [ ]:
import json
from pathlib import Path

release = json.loads(Path('/content/flynet-release/results/release.json').read_text())
print(json.dumps({
    'provenance': release.get('provenance'),
    'artifacts': release.get('files', release.get('artifacts')),
    'verification': release.get('verification'),
}, indent=2, sort_keys=True))

## 3. Smoke-test the model in a game

Every selected action is checked by `python-chess`; the decision readout exposes candidates and compact activity telemetry.

In [ ]:
!python -m flychess.cli \
  --flynet-weights /content/flynet-release/flynet.safetensors \
  --flynet-graph /content/flynet-release/flynet-graph.npz \
  --flynet-graph-metadata /content/flynet-release/flynet-graph.json \
  --flynet-config /content/flynet-release/flynet-config.json \
  --engine stockfish --depth 3 --max-plies 40

## 4. Publish the model-only artifact

Run this only after the source commit is pushed and the evaluation receipt has been reviewed. The Hub payload contains the weights, graph, config, optional labeled dataset, training history, model card, and release checksums; it does not copy the source tree.

In [ ]:
# Replace the placeholder with the intended Hub namespace before running.
!python scripts/publish_flynet_model.py \
  --source-dir /content/flychess \
  --release-dir /content/flynet-release \
  --dataset /content/flynet-dataset.npz \
  --repo-id YOUR_HF_NAMESPACE/flychess \
  --prune